In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/UChicago/TTIC31040/homography")

Mounted at /content/drive


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from multiprocessing import Pool, cpu_count
import pandas as pd

### 1. Helper functions


In [ ]:
# Reprojection Error
def reprojection_error(H, src_pts, dst_pts):
  src_h = cv2.convertPointsToHomogeneous(src_pts)[:,0,:]
  proj = (H @ src_h.T).T
  proj = proj[:, :2] / proj[:, 2][:, np.newaxis]

  error = np.linalg.norm(proj - dst_pts.reshape(-1,2), axis=1)

  return np.mean(error)


# Filter keypoints: whole, center, border
def filter_keys(kp, des, image_shape, region, threshold=0.2):
  if region == "whole":
      return kp, des

  h, w = image_shape

  coords = np.array([k.pt for k in kp])

  center_mask = (
      (coords[:,0] > w*threshold) &
      (coords[:,0] < w*(1-threshold)) &
      (coords[:,1] > h*threshold) &
      (coords[:,1] < h*(1-threshold))
  )

  if region == "center":
      mask = center_mask
  else:
      mask = ~center_mask

  selected_kp = [kp[i] for i in np.where(mask)[0]]
  selected_des = des[mask]

  if len(selected_kp) < 4:
      return [], None

  return selected_kp, selected_des


# SIFT
def sift(image):
  gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  sift = cv2.SIFT_create()
  kp, des = sift.detectAndCompute(gray, None)
  return gray, kp, des


# Matching keypoints
def match_key(des1, des2, ratio=0.7):
  bf = cv2.BFMatcher(cv2.NORM_L2)

  matches = bf.knnMatch(des1, des2, k=2)

  good = []

  for m, n in matches:
      if m.distance < ratio * n.distance:
          good.append(m)

  return good


# RANSAC
def homography(kp1, kp2, matches):
  if len(matches) < 4:
      return None, None, None, None

  src_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1,1,2)
  dst_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1,1,2)

  H, inliers = cv2.findHomography(
      src_pts,
      dst_pts,
      cv2.RANSAC,
      3.0,
      maxIters=5000,
      confidence=0.995
  )

  if H is None:
      return None, None, None, None

  src_in = src_pts[inliers.ravel()==1]
  dst_in = dst_pts[inliers.ravel()==1]
  return H, src_in, dst_in, inliers


### 2. Run images pairs for one shared reference

In [ ]:
# Pair
def process_pair(folder, distortion, kp_ref, des_ref, kp_tar, des_tar, ref_shape):
  rows = []

  for region in ["whole","center","border"]:
      kp_sel, des_sel = filter_keys(
          kp_ref,
          des_ref,
          ref_shape,
          region
      )

      if des_sel is None:
        rows.append({
            "img": folder,
            "distortion": distortion,
            "region": region,
            "H": None,
            "num_keypoints": 0,
            "num_matches": 0,
            "num_inliers": 0,
            "reprojection_error": None
        })
        continue

      matches = match_key(des_sel, des_tar)

      H, src_in, dst_in, inliers = homography(
          kp_sel,
          kp_tar,
          matches
      )

      if H is None:
        rows.append({
            "img": folder,
            "distortion": distortion,
            "region": region,
            "H": None,
            "num_keypoints": len(kp_sel),
            "num_matches": len(matches),
            "num_inliers": 0,
            "reprojection_error": None
        })
        continue

      error = reprojection_error(H, src_in, dst_in)

      rows.append({
          "img": folder,
          "distortion": distortion,
          "region": region,
          "H": H.tolist(),
          "num_keypoints": len(kp_sel),
          "num_matches": len(matches),
          "num_inliers": int(inliers.sum()),
          "reprojection_error": error
      })

  return rows


# Folder
def process_folder(folder_path):
  folder = os.path.basename(folder_path)

  ref = cv2.imread(os.path.join(folder_path,"1.ppm"))
  small = cv2.imread(os.path.join(folder_path,"2.ppm"))
  large = cv2.imread(os.path.join(folder_path,"6.ppm"))

  ref_gray, kp_ref, des_ref = sift(ref)

  _, kp_small, des_small = sift(small)
  _, kp_large, des_large = sift(large)

  rows = []

  rows += process_pair(
      folder,"small",
      kp_ref,des_ref,
      kp_small,des_small,
      ref_gray.shape
  )

  rows += process_pair(
      folder,"large",
      kp_ref,des_ref,
      kp_large,des_large,
      ref_gray.shape
  )

  return rows

In [ ]:
# Parallel data processing
def run_dataset_parallel(dataset_path):

    folders = [
        os.path.join(dataset_path,f)
        for f in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path,f))
    ]

    with Pool(cpu_count()) as p:
        results = p.map(process_folder, folders)

    rows = [r for folder in results for r in folder]

    df = pd.DataFrame(rows)

    return df

In [ ]:
dataset_path = "./hpatches-sequences-release"

df = run_dataset_parallel(dataset_path)

df.to_csv("./homography_results.csv", index=False)

print("Finished.")

Finished.
